# Setup

In [ ]:
import jax
from jax.experimental import pallas as pl
import jax.numpy as jnp
import numpy as np
from functools import partial
from jax.experimental.pallas import tpu as pltpu
from dataclasses import dataclass
from jax import random
from typing import Tuple

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [ ]:
BACKEND = jax.default_backend()
DEVICES = jax.devices()

IS_CPU = BACKEND == "cpu"
IS_TPU = BACKEND == "tpu"
IS_GPU = BACKEND == "gpu"

# Pass this into pl.pallas_call(..., interpret=PALLAS_INTERPRET)
PALLAS_INTERPRET = IS_CPU

print("JAX backend:", BACKEND)
print("Devices:", DEVICES)
print("PALLAS_INTERPRET:", PALLAS_INTERPRET)

pallas_call = partial(pl.pallas_call, interpret=PALLAS_INTERPRET)

JAX backend: tpu
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
PALLAS_INTERPRET: False


In [ ]:
!pip install -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

Looking in links: https://storage.googleapis.com/jax-releases/libtpu_releases.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 103.0 MB/s eta 0:00:00
  Attempting uninstall: libtpu
    Found existing installation: libtpu 0.0.21
    Uninstalling libtpu-0.0.21:
      Successfully uninstalled libtpu-0.0.21
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.7.2
    Uninstalling jaxlib-0.7.2:
      Successfully uninstalled jaxlib-0.7.2
  Attempting uninstall: jax
    Found existing installation: jax 0.7.2
    Uninstalling jax-0.7.2:
      Successfully uninstalled jax-0.7.2


# Add 1 Kernel

## Implementation

In [ ]:
def add1_kernel(x_ref, y_ref):
    y_ref[...] = x_ref[...] + 1

def add1(x, bm, bn):
    """
    x: (m,n)
    bm: m block size
    bn: n block size
    """
    m, n = x.shape
    in_out_specs = [
        pl.BlockSpec((bm, bn), lambda i,j: (i,j))
    ]
    grid = (m // bm, n // bn)
    return pallas_call(
        add1_kernel,
        out_shape=jax.ShapeDtypeStruct((m, n), x.dtype),
        in_specs=in_out_specs,
        out_specs=in_out_specs[0],
        grid=grid,
    )(x)

## Tests

In [ ]:
def check_add1(shape, bm, bn, dtype=jnp.float32):
    m, n = shape
    assert m % bm == 0, (m, bm)
    assert n % bn == 0, (n, bn)

    x = jnp.arange(m * n, dtype=dtype).reshape(m, n)
    expected = x + jnp.array(1, dtype=dtype)

    got = add1(x, bm, bn)
    got.block_until_ready()

    np.testing.assert_allclose(
        np.asarray(got),
        np.asarray(expected),
        rtol=0,
        atol=0,
    )

    max_err = jnp.max(jnp.abs(got.astype(jnp.float32) - expected.astype(jnp.float32)))
    print(f"PASS shape={shape}, block=({bm},{bn}), dtype={dtype}, max_err={max_err}")

# one-block case
check_add1((8, 128), bm=8, bn=128)

# Multiple blocks
check_add1((16, 256), bm=8, bn=128)
check_add1((32, 512), bm=8, bn=128)

# Different legal block shapes
check_add1((16, 128), bm=16, bn=128)
check_add1((8, 256), bm=8, bn=256)

# dtype checks
check_add1((16, 256), bm=8, bn=128, dtype=jnp.int32)
check_add1((16, 256), bm=8, bn=128, dtype=jnp.bfloat16)

print("All add1 correctness checks passed.")

PASS shape=(8, 128), block=(8,128), dtype=<class 'jax.numpy.float32'>, max_err=0.0
PASS shape=(16, 256), block=(8,128), dtype=<class 'jax.numpy.float32'>, max_err=0.0
PASS shape=(32, 512), block=(8,128), dtype=<class 'jax.numpy.float32'>, max_err=0.0
PASS shape=(16, 128), block=(16,128), dtype=<class 'jax.numpy.float32'>, max_err=0.0
PASS shape=(8, 256), block=(8,256), dtype=<class 'jax.numpy.float32'>, max_err=0.0
PASS shape=(16, 256), block=(8,128), dtype=<class 'jax.numpy.int32'>, max_err=0.0
PASS shape=(16, 256), block=(8,128), dtype=<class 'jax.numpy.bfloat16'>, max_err=0.0
All add1 correctness checks passed.


# Mat Mul Kernel

## Implementation

In [ ]:
def matmul_kernel(x_ref, y_ref, z_ref, acc_ref, num_reduction_steps):
    @pl.when(pl.program_id(2) == 0)
    def _():
        acc_ref[...] = jnp.zeros_like(acc_ref)

    acc_ref[...] += jnp.dot(
        x_ref[...], y_ref[...], preferred_element_type=jnp.float32
    )

    @pl.when(pl.program_id(2) == num_reduction_steps-1)
    def _():
        z_ref[...] = acc_ref[...].astype(z_ref.dtype)

def matmul(
    x,
    y,
    bm,
    bk,
    bn,
):
    """
    x: (m, k)
    y: (k, n)
    bm: m block size
    bk: k block size
    bn: n block size
    """
    m, k = x.shape
    _, n = y.shape

    assert x.shape[-1] == y.shape[0]

    grid = (m // bm, n // bn, k // bk)
    in_specs = (
        pl.BlockSpec((bm, bk), lambda i,j,k: (i,k)),
        pl.BlockSpec((bk, bn), lambda i,j,k: (k, j))
    )
    out_specs = pl.BlockSpec((bm, bn), lambda i,j,k: (i,j))
    out_shape = jax.ShapeDtypeStruct((m, n), x.dtype)
    grid_spec = pltpu.PrefetchScalarGridSpec(
        num_scalar_prefetch=0,
        grid=grid,
        in_specs=in_specs,
        out_specs=out_specs,
        scratch_shapes=[pltpu.VMEM((bm, bn), jnp.float32)],
    )
    compiler_params = pltpu.CompilerParams(
        dimension_semantics=("parallel", "parallel", "arbitrary")
    )
    kernel_fn = partial(matmul_kernel, num_reduction_steps=(k // bk))
    return pallas_call(
        kernel_fn,
        grid_spec=grid_spec,
        out_shape=out_shape,
        compiler_params=compiler_params,
    )(x,y)


## Tests

In [ ]:
def check_matmul(shape_x, shape_y, bm, bk, bn, dtype=jnp.float32, rtol=None, atol=None):
    m, k = shape_x
    k2, n = shape_y
    assert k == k2, (shape_x, shape_y)
    assert m % bm == 0, (m, bm)
    assert k % bk == 0, (k, bk)
    assert n % bn == 0, (n, bn)

    key = jax.random.key(0)
    kx, ky = jax.random.split(key)

    if dtype in (jnp.int32, jnp.int64):
        x = jax.random.randint(kx, shape_x, minval=-5, maxval=5, dtype=dtype)
        y = jax.random.randint(ky, shape_y, minval=-5, maxval=5, dtype=dtype)
    else:
        x = jax.random.normal(kx, shape_x, dtype=dtype)
        y = jax.random.normal(ky, shape_y, dtype=dtype)

    got = matmul(x, y, bm=bm, bk=bk, bn=bn)
    expected = x @ y

    got.block_until_ready()

    if rtol is None or atol is None:
        if dtype == jnp.bfloat16:
            rtol, atol = 2e-2, 2e-2
        elif dtype == jnp.float16:
            rtol, atol = 2e-2, 2e-2
        else:
            rtol, atol = 1e-5, 1e-5

    np.testing.assert_allclose(
        np.asarray(got),
        np.asarray(expected),
        rtol=rtol,
        atol=atol,
    )

    max_err = jnp.max(jnp.abs(got.astype(jnp.float32) - expected.astype(jnp.float32)))
    print(
        f"PASS x={shape_x}, y={shape_y}, block=({bm},{bk},{bn}), "
        f"dtype={dtype}, max_err={max_err}"
    )

# One output block, one K block
check_matmul((8, 128), (128, 128), bm=8, bk=128, bn=128)

# Multiple M/N blocks, one K block
check_matmul((16, 128), (128, 256), bm=8, bk=128, bn=128)

# Multiple K blocks
check_matmul((16, 256), (256, 128), bm=8, bk=128, bn=128)

# Multiple M, K, and N blocks
check_matmul((32, 256), (256, 256), bm=8, bk=128, bn=128)

# Different block shape
check_matmul((16, 256), (256, 256), bm=16, bk=128, bn=128)

# bf16, if your kernel supports it
check_matmul((16, 256), (256, 256), bm=8, bk=128, bn=128, dtype=jnp.bfloat16)

print("All matmul correctness checks passed.")

PASS x=(8, 128), y=(128, 128), block=(8,128,128), dtype=<class 'jax.numpy.float32'>, max_err=0.0
PASS x=(16, 128), y=(128, 256), block=(8,128,128), dtype=<class 'jax.numpy.float32'>, max_err=1.9073486328125e-05
PASS x=(16, 256), y=(256, 128), block=(8,128,128), dtype=<class 'jax.numpy.float32'>, max_err=2.6702880859375e-05
PASS x=(32, 256), y=(256, 256), block=(8,128,128), dtype=<class 'jax.numpy.float32'>, max_err=3.4332275390625e-05
PASS x=(16, 256), y=(256, 256), block=(16,128,128), dtype=<class 'jax.numpy.float32'>, max_err=2.6702880859375e-05
PASS x=(16, 256), y=(256, 256), block=(8,128,128), dtype=<class 'jax.numpy.bfloat16'>, max_err=0.0
All matmul correctness checks passed.


# Dense FFN Kernel

In [ ]:
def ffn_kernel(
    x_ref,
    w1_ref,
    w2_ref,
    w_down_ref,
    out_ref,
    w1_acc_ref,
    w2_acc_ref,
    out_acc_ref, *,
    reduction_end_idxs,
):
    f_axis = 2
    d_in_axis = 3
    final_f_idx, final_d_in_idx = reduction_end_idxs

    @pl.when(pl.program_id(d_in_axis) == 0)
    def _():
        w1_acc_ref[...] = jnp.zeros_like(w1_acc_ref)
        w2_acc_ref[...] = jnp.zeros_like(w2_acc_ref)

    @pl.when(pl.program_id(f_axis) == 0)
    def _():
        out_acc_ref[...] = jnp.zeros_like(out_acc_ref)

    w1_acc_ref[...] += jnp.dot(
        x_ref[...], w1_ref[...], preferred_element_type=jnp.float32
    )

    w2_acc_ref[...] += jnp.dot(
        x_ref[...], w2_ref[...], preferred_element_type=jnp.float32
    )

    @pl.when(pl.program_id(d_in_axis) == final_d_in_idx)
    def _():
        out_acc_ref[...] += jnp.dot(
            jax.nn.swish(w1_acc_ref[...].astype(w1_ref.dtype))*w2_acc_ref[...].astype(w2_ref.dtype), w_down_ref[...],
            preferred_element_type=jnp.float32,
        )

    @pl.when(
        (pl.program_id(f_axis) == final_f_idx) & (pl.program_id(d_in_axis) == final_d_in_idx)
    )
    def _():
        out_ref[...] = out_acc_ref[...].astype(out_ref.dtype)


def fused_ffn(
    x,
    w1,
    w2,
    w_down,
    bn,
    bd_in,
    bf,
    bd_out,
):
    """
    x: (n, d)
    w1: (d, f)
    w2: (d, f)
    w_down: (f, d)
    bn: n block size
    bd_in: d block size for w1/w2
    bf: f block size
    bd_out: d block size for w_down
    """
    n, d_in = x.shape
    _, f = w1.shape
    _, d_out = w_down.shape

    grid = (n // bn, d_out // bd_out, f // bf, d_in // bd_in)
    in_specs = [
        pl.BlockSpec((bn, bd_in), lambda i,j,k,l: (i,l)),
        pl.BlockSpec((bd_in, bf), lambda i,j,k,l: (l, k)),
        pl.BlockSpec((bd_in, bf), lambda i,j,k,l: (l, k)),
        pl.BlockSpec((bf, bd_out), lambda i,j,k,l: (k, j)),
    ]
    out_specs = pl.BlockSpec((bn, bd_out), lambda i,j,k,l: (i, j))
    out_shape = jax.ShapeDtypeStruct((n, d_out), x.dtype)
    scratch_shapes = [
        pltpu.VMEM((bn, bf), jnp.float32),
        pltpu.VMEM((bn, bf), jnp.float32),
        pltpu.VMEM((bn, bd_out), jnp.float32),
    ]
    grid_spec = pltpu.PrefetchScalarGridSpec(
        num_scalar_prefetch=0,
        grid=grid,
        in_specs=in_specs,
        out_specs=out_specs,
        scratch_shapes=scratch_shapes,
    )
    compiler_params = pltpu.CompilerParams(
        dimension_semantics=("parallel", "parallel", "arbitrary", "arbitrary")
    )
    reduction_end_idxs = (
        (f // bf) - 1,
        (d_in // bd_in) - 1,
    )
    kernel_fn = partial(ffn_kernel, reduction_end_idxs=reduction_end_idxs)
    return pallas_call(
        kernel_fn,
        grid_spec=grid_spec,
        out_shape=out_shape,
        compiler_params=compiler_params,
    )(x, w1, w2, w_down)

## Tests

In [ ]:
def check_ffn(n, d, f, bn, bd_in, bf, bd_out,
              dtype=jnp.float32, rtol=None, atol=None):
    assert n % bn == 0,     (n, bn)
    assert d % bd_in == 0,  (d, bd_in)    # up-proj contraction axis
    assert f % bf == 0,     (f, bf)       # down-proj contraction axis
    assert d % bd_out == 0, (d, bd_out)   # down-proj output axis

    key = jax.random.key(0)
    kx, k1, k2, kd = jax.random.split(key, 4)


    x  = jax.random.normal(kx, (n, d), dtype=dtype)                       # [N, D]
    W1 = jax.random.normal(k1, (d, f), dtype=dtype) / np.sqrt(d)          # [D, F] up
    W2 = jax.random.normal(k2, (d, f), dtype=dtype) / np.sqrt(d)          # [D, F] gate
    Wd = jax.random.normal(kd, (f, d), dtype=dtype) / np.sqrt(f)          # [F, D] down


    got = fused_ffn(x, W1, W2, Wd, bn=bn, bd_in=bd_in, bf=bf, bd_out=bd_out)


    # fp32 reference from the SAME (possibly bf16) inputs, so input quantization
    # is shared and we isolate the kernel's compute error.
    xf, W1f, W2f, Wdf = (a.astype(jnp.float32) for a in (x, W1, W2, Wd))
    h = jax.nn.swish(xf @ W1f) * (xf @ W2f)
    expected = h @ Wdf

    got.block_until_ready()

    if rtol is None or atol is None:
        if dtype in (jnp.bfloat16, jnp.float16):
            rtol, atol = 3e-2, 3e-2
        else:
            rtol, atol = 1e-4, 1e-4

    got32 = np.asarray(got, dtype=np.float32)
    exp32 = np.asarray(expected, dtype=np.float32)
    np.testing.assert_allclose(got32, exp32, rtol=rtol, atol=atol)

    max_err = float(np.max(np.abs(got32 - exp32)))
    print(f"PASS n={n}, d={d}, f={f}, "
          f"block=(bn={bn},bd_in={bd_in},bf={bf},bd_out={bd_out}), "
          f"dtype={dtype.__name__}, max_err={max_err:.3e}")


# 1. single tile on every axis — smoke test
check_ffn(n=8,  d=128, f=128, bn=8, bd_in=128, bf=128, bd_out=128)

# 2. multiple batch (n) blocks
check_ffn(n=32, d=128, f=128, bn=8, bd_in=128, bf=128, bd_out=128)

# 3. multiple f blocks
check_ffn(n=8,  d=128, f=512, bn=8, bd_in=128, bf=128, bd_out=128)

# 4. multiple d_in blocks
#    d_out stays a single tile (bd_out == d)
check_ffn(n=8,  d=256, f=128, bn=8, bd_in=128, bf=128, bd_out=256)

# 5. multiple d_out blocks
#    d_in stays a single tile (bd_in == d)
check_ffn(n=8,  d=256, f=128, bn=8, bd_in=256, bf=128, bd_out=128)

# 6. everything tiled at once
check_ffn(n=32, d=256, f=512, bn=8, bd_in=128, bf=128, bd_out=128)

# 7. bf16
check_ffn(n=32, d=256, f=512, bn=8, bd_in=128, bf=128, bd_out=128, dtype=jnp.bfloat16)

print("All FFN correctness checks passed.")

PASS n=8, d=128, f=128, block=(bn=8,bd_in=128,bf=128,bd_out=128), dtype=float32, max_err=0.000e+00
PASS n=32, d=128, f=128, block=(bn=8,bd_in=128,bf=128,bd_out=128), dtype=float32, max_err=9.537e-07
PASS n=8, d=128, f=512, block=(bn=8,bd_in=128,bf=128,bd_out=128), dtype=float32, max_err=6.258e-07
PASS n=8, d=256, f=128, block=(bn=8,bd_in=128,bf=128,bd_out=256), dtype=float32, max_err=1.192e-06
PASS n=8, d=256, f=128, block=(bn=8,bd_in=256,bf=128,bd_out=128), dtype=float32, max_err=0.000e+00
PASS n=32, d=256, f=512, block=(bn=8,bd_in=128,bf=128,bd_out=128), dtype=float32, max_err=1.550e-06
PASS n=32, d=256, f=512, block=(bn=8,bd_in=128,bf=128,bd_out=128), dtype=bfloat16, max_err=7.681e-03
All FFN correctness checks passed.


# Real Kernel Playground

## Non kernel version

In [ ]:
def _grad_rhs_ragged(lhs, g, group_sizes, num_groups):
    rdn = jax.lax.RaggedDotDimensionNumbers(
        dot_dimension_numbers=(((0,), (0,)), ((), ())),
        lhs_ragged_dimensions=(0,), rhs_group_dimensions=())
    return jax.lax.ragged_dot_general(lhs, g, group_sizes, ragged_dot_dimension_numbers=rdn)

def _grad_rhs_segment(lhs, g, group_sizes, num_groups):
    M = lhs.shape[0]
    tg = jnp.repeat(jnp.arange(num_groups), group_sizes, total_repeat_length=M)
    return jax.ops.segment_sum(lhs[:, :, None] * g[:, None, :], tg, num_segments=num_groups)

_grad_rhs = _grad_rhs_ragged

@jax.custom_vjp
def grouped_matmul(lhs, rhs, group_sizes):
    return jax.lax.ragged_dot(lhs, rhs, group_sizes)

def _gmm_fwd(lhs, rhs, group_sizes):
    return jax.lax.ragged_dot(lhs, rhs, group_sizes), (lhs, rhs, group_sizes)

def _gmm_bwd(res, g):
    lhs, rhs, group_sizes = res
    grad_lhs = jax.lax.ragged_dot(g, jnp.swapaxes(rhs, 1, 2), group_sizes)
    grad_rhs = _grad_rhs(lhs, g, group_sizes, rhs.shape[0])
    return (grad_lhs, grad_rhs, None)

grouped_matmul.defvjp(_gmm_fwd, _gmm_bwd)


@dataclass(frozen=True)
class MoEFFN:
    num_experts: int
    k: int
    model_dim: int
    up_proj: int

    def init(self, key):
        key, *subkeys = random.split(key, num=5)
        bound = jnp.sqrt(3) / jnp.sqrt(self.model_dim)
        W1 = random.uniform(
            subkeys[0],
            shape=(self.num_experts, self.model_dim, self.up_proj),
            dtype=jnp.float32,
            minval=-0.4*bound,
            maxval=0.4*bound,
        )
        W2 = random.uniform(
            subkeys[1],
            shape=(self.num_experts, self.model_dim, self.up_proj),
            dtype=jnp.float32,
            minval=-0.4*bound,
            maxval=0.4*bound,
        )
        W_down = jnp.zeros(
            shape=(self.num_experts, self.up_proj, self.model_dim),
            dtype=jnp.float32,
        )

        router = random.normal(
            key=subkeys[3],
            shape=(self.model_dim, self.num_experts),
            dtype=jnp.float32,
        )*0.02

        return {
            'W1': W1,
            'W2': W2,
            'W_down': W_down,
            'router': router,
        }

    def apply(self, params, x, pad_mask):
        """
        x: BTD
        """
        # TODO: compute and propogate aux loss
        e = self.num_experts
        k = self.k
        b, t, d = x.shape
        n = b*t
        router_params = params['router']
        W1 = params['W1'] # EDF
        W2 = params['W2'] # EDF

        W_down = params['W_down'] # EFD

        # reshape for whole layer, and then reshape back at end
        x = jnp.reshape(x, (n,d))
        pad_mask = jnp.reshape(pad_mask, (n,)).astype(jnp.float32)


        # compute router logits in fp32
        router_logits = jnp.einsum('nd,de->ne', x.astype(jnp.float32), router_params) # NE

        # compute z loss
        z_loss = jnp.sum(jnp.power(jax.nn.logsumexp(router_logits, axis=1), 2)*pad_mask)/jnp.sum(pad_mask)

        router_probs = jax.nn.softmax(router_logits, axis=-1) # NE
        top_k_values, top_k_idxs = jax.lax.top_k(router_probs, k=k) # Nk


        # flatten and argsort the indices
        top_k_idxs = jnp.reshape(top_k_idxs, (n*k,))
        argsorted_idxs = jnp.argsort(top_k_idxs)
        expert_contiguous_x = x[argsorted_idxs // k] # (Nk)D
        # explicitly cast to int32
        group_sizes = jnp.bincount(top_k_idxs, length=e).astype(jnp.int32) # E

        # compute f and p
        num_token_assignments = jnp.sum(pad_mask)*k
        f = jnp.bincount(top_k_idxs, weights=jnp.repeat(pad_mask, k, total_repeat_length=n*k), length=e) / num_token_assignments
        p = jnp.sum(router_probs*pad_mask[:,None], axis=0) / jnp.sum(pad_mask)


        # route tokens through respective expert FFNs
        w1_out = grouped_matmul(expert_contiguous_x, W1.astype(jnp.bfloat16), group_sizes)
        w2_out = grouped_matmul(expert_contiguous_x, W2.astype(jnp.bfloat16), group_sizes)
        swiglu_x = jax.nn.swish(w1_out)*w2_out # (kN)F
        out    = grouped_matmul(swiglu_x,            W_down.astype(jnp.bfloat16), group_sizes)
        # unscramble and aggregate
        unperm = jnp.argsort(argsorted_idxs)
        out = jnp.reshape(out[unperm], (n, k, d)) # NkD
        out = jnp.einsum('nkd,nk->nd', out, top_k_values.astype(jnp.bfloat16)) # ND

        out = jnp.reshape(out, (b, t, d))
        return out, {
            'z': z_loss,
            'f': f,
            'p': p
        }

## Playground

### Get Metadata Function

In [ ]:
# The functions in this cell are borrowed from
# https://github.com/jax-ml/jax/blob/main/jax/experimental/pallas/ops/tpu/megablox/gmm.py

def _calculate_num_tiles(x: int, tx: int) -> int:
    tiles, rem = divmod(x, tx)
    if rem:
        raise ValueError(f"{x} must be divisible by x-dimension tile size ({tx}).")
    return tiles

def _calculate_irregular_num_tiles(x: int, tx: int) -> tuple[int, int]:
    tiles, rem = divmod(x, tx)
    if rem:
        tiles += 1
    return tiles, rem

def make_group_metadata(
    *,
    group_sizes: jnp.ndarray,
    m: int,
    tm: int,
    start_group: jnp.ndarray,
    num_nonzero_groups: int,
    visit_empty_groups: bool = True,
) -> Tuple:
  """Create the metadata needed for grouped matmul computation.

  Args:
    group_sizes: A 1d, jnp.ndarray with shape [num_groups] and jnp.int32 dtype.
    m: The number of rows in lhs.
    tm: The m-dimension tile size being used.
    start_group: The group in group sizes to start computing from. This is
      particularly useful for when rhs num_groups is sharded.
    num_nonzero_groups: Number of groups in group sizes to compute on. Useful in
      combination with group_offset.
    visit_empty_groups: If True, do not squeeze tiles for empty groups out of
      the metadata. This is necessary for tgmm, where we at least need to zero
      the output for each group.

  Returns:
    tuple of:
      group_offsets: A 1d, jnp.ndarray with shape [num_groups+1] and jnp.int32
        dtype. group_offsets[i] indicates the row at which group [i] starts in
        the lhs matrix and group_offsets[i-1] = m.
      group_ids: A 1d, jnp.ndarray with shape [m_tiles + num_groups] and
        jnp.int32 dtype. group_ids[i] indicates which group grid index 'i' will
        work on.
      m_tile_ids: A 1d, jnp.ndarray with shape [m_tiles + num_groups] and
        jnp.int32. m_tile_ids[i] indicates which m-dimension tile grid index 'i'
        will work on.
    num_tiles: The number of m-dimension tiles to execute.
  """
  num_groups = group_sizes.shape[0]
  end_group = start_group + num_nonzero_groups - 1

  # Calculate the offset of each group, starting at zero. This metadata is
  # similar to row offsets in a CSR matrix. The following properties hold:
  #
  # group_offsets.shape = [num_groups + 1]
  # group_offsets[0] = 0
  # group_offsets[num_groups] = m
  #
  # The row at which group 'i' starts is group_offsets[i].
  group_ends = jnp.cumsum(group_sizes)
  group_offsets = jnp.concatenate([jnp.zeros(1, dtype=jnp.int32), group_ends])

  # Assign a group id to each grid index.
  #
  # If a group starts somewhere other than the start of a tile or ends somewhere
  # other than the end of a tile we need to compute that full tile. Calculate
  # the number of tiles for each group by rounding their end up to the nearest
  # 'tm' and their start down to the nearest 'tm'.

  # (1) Round the group_ends up to the nearest multiple of 'tm'.
  #
  # NOTE: This does not change group_offsets[num_groups], which is m
  # (because we enforce m is divisible by tm).
  rounded_group_ends = ((group_ends + tm - 1) // tm * tm).astype(jnp.int32)

  # (2) Round the group_starts down to the nearest multiple of 'tm'.
  group_starts = jnp.concatenate(
      [jnp.zeros(1, dtype=jnp.int32), group_ends[:-1]]
  )
  rounded_group_starts = group_starts // tm * tm

  # (3) Calculate the number of rows in each group.
  #
  # NOTE: Handle zero-sized groups as a special case. If the start for a
  # zero-sized group is not divisible by 'tm' its start will be rounded down and
  # its end will be rounded up such that its size will become 1 tile here.
  rounded_group_sizes = rounded_group_ends - rounded_group_starts
  rounded_group_sizes = jnp.where(group_sizes == 0, 0, rounded_group_sizes)

  # (4) Convert the group sizes from units of rows to unit of 'tm' sized tiles.
  #
  # An m-dimension tile is 'owned' by group 'i' if the first row of the tile
  # belongs to group 'i'. In addition to owned tiles, each group can have 0 or 1
  # initial partial tiles if it's first row does not occur in the first row of a
  # tile. The '0-th' group never has a partial tile because it always starts at
  # the 0-th row.
  #
  # If no group has a partial tile, the total number of tiles is equal to
  # 'm // tm'. If every group has a partial except the 0-th group, the total
  # number of tiles is equal to 'm // tm + num_groups - 1'. Thus we know that
  #
  # tiles_m <= group_tiles.sum() <= tiles_m + num_groups - 1
  #
  # Where tiles_m = m // tm.
  #
  # NOTE: All group sizes are divisible by 'tm' because of the rounding in steps
  # (1) and (2) so this division is exact.
  group_tiles = rounded_group_sizes // tm

  if visit_empty_groups:
    # Insert one tile for empty groups.
    group_tiles = jnp.where(group_sizes == 0, 1, group_tiles)

  # Create the group ids for each grid index based on the tile counts for each
  # group.
  #
  # NOTE: This repeat(...) will pad group_ids with the final group id if
  # group_tiles.sum() < tiles_m + num_groups - 1. The kernel grid will be sized
  # such that we only execute the necessary number of tiles.
  tiles_m = _calculate_num_tiles(m, tm)
  group_ids = jnp.repeat(
      jnp.arange(num_groups, dtype=jnp.int32),
      group_tiles,
      total_repeat_length=tiles_m + num_groups - 1,
  )

  # Assign an m-dimension tile id to each grid index.
  #
  # NOTE: Output tiles can only be re-visited consecutively. The following
  # procedure guarantees that m-dimension tile indices respect this.

  # (1) Calculate how many times each m-dimension tile will be visited.
  #
  # Each tile is guaranteed to be visited once by the group that owns the tile.
  # The remaining possible visits occur when a group starts inside of a tile at
  # a position other than the first row. We can calculate which m-dimension tile
  # each group starts in by floor-dividing its offset with `tm` and then count
  # tile visits with a histogram.
  #
  # To avoid double counting tile visits from the group that owns the tile,
  # filter these out by assigning their tile id to `tile_m` (one beyond the max)
  # such that they're ignored by the subsequent histogram. Also filter out any
  # group which is empty.
  #
  # TODO(tgale): Invert the 'partial_tile_mask' predicates to be more clear.
  partial_tile_mask = jnp.logical_or(
      (group_offsets[:-1] % tm) == 0, group_sizes == 0
  )

  # Explicitly enable tiles for zero sized groups, if specified. This covers
  # zero sized groups that start on a tile-aligned row and those that do not.
  if visit_empty_groups:
    partial_tile_mask = jnp.where(group_sizes == 0, 0, partial_tile_mask)

  partial_tile_ids = jnp.where(
      partial_tile_mask, tiles_m, group_offsets[:-1] // tm
  )

  tile_visits = (
      jnp.histogram(partial_tile_ids, bins=tiles_m, range=(0, tiles_m - 1))[0]
      + 1
  )

  # Create the m-dimension tile ids for each grid index based on the visit
  # counts for each tile.
  m_tile_ids = jnp.repeat(
      jnp.arange(tiles_m, dtype=jnp.int32),
      tile_visits.astype(jnp.int32),
      total_repeat_length=tiles_m + num_groups - 1,
  )

  # Account for sharding.
  #
  # Find the start of the groups owned by our shard and shift the group_ids and
  # m_tile_ids s.t. the metadata for our tiles are at the front of the arrays.
  #
  # TODO(tgale): Move this offset into the kernel to avoid these rolls.
  first_tile_in_shard = (group_ids < start_group).sum()
  group_ids = jnp.roll(group_ids, shift=-first_tile_in_shard, axis=0)
  m_tile_ids = jnp.roll(m_tile_ids, shift=-first_tile_in_shard, axis=0)

  # Calculate the number of tiles we need to compute for our shard.
  #
  # Remove tile visits that belong to a group not in our shard.
  iota = jnp.arange(num_groups, dtype=jnp.int32)
  active_group_mask = jnp.logical_and(iota <= end_group, iota >= start_group)
  group_tiles = jnp.where(active_group_mask, group_tiles, 0)
  num_tiles = group_tiles.sum()
  return (group_offsets, group_ids, m_tile_ids), num_tiles

### Core Kernel Logic

In [ ]:
def kernel(
        metadata,    # prefetch
        lhs_ref,     # input ref
        w1_ref,      # input ref
        w2_ref,      # input ref
        w_down_ref,  # input ref
        out_ref,     # output ref
        w1_acc_ref,  # scratch ref
        w2_acc_ref,  # scratch ref
        out_acc_ref, # scratch ref
        *,
        k_rem,
        n_rem,
        bm,
        reduction_end_idxs,
):
    (group_offsets, group_ids, m_tile_ids) = metadata
    last_n_idx, last_k_idx = reduction_end_idxs
    n_tile = pl.program_id(2)
    is_last_n_tile = pl.program_id(2) == last_n_idx
    k_tile = pl.program_id(3)
    is_last_k_tile = pl.program_id(3) == last_k_idx
    i = pl.program_id(1)
    m_tile = m_tile_ids[i]
    group = group_ids[i]


    """
    masks to take care of:
        - masking out m rows
        - masking out k, n rows for accumulation
    """


    @pl.when(k_tile == 0)
    def zero_up_accs():
        w1_acc_ref[...] = jnp.zeros_like(w1_acc_ref)
        w2_acc_ref[...] = jnp.zeros_like(w2_acc_ref)

    @pl.when(n_tile == 0)
    def zero_out_acc():
        out_acc_ref[...] = jnp.zeros_like(out_acc_ref)

    def m_tile_mask():
        start_idx = group_offsets[group] - bm*m_tile
        end_idx = group_offsets[group+1] - bm*m_tile

        iota = jax.lax.broadcasted_iota(
            shape=out_ref.shape,
            dtype=jnp.int32,
            dimension=0
        )
        return jnp.logical_and(iota >= start_idx, iota < end_idx)

    def k_tile_masked(x, dim):
        if k_rem > 0:
            iota = jax.lax.broadcasted_iota(
                shape=x.shape,
                dtype=jnp.int32,
                dimension=dim,
            )
            mask = (iota < k_rem) | (~is_last_k_tile)
        else:
            mask = jnp.full(x.shape, True)

        return jnp.where(mask, x, 0)

    def n_tile_masked(x, dim):
        if n_rem > 0:
            iota = jax.lax.broadcasted_iota(
                shape=x.shape,
                dtype=jnp.int32,
                dimension=dim,
            )
            mask = (iota < n_rem) | (~is_last_n_tile)
        else:
            mask = jnp.full(x.shape, True)

        return jnp.where(mask, x, 0)


    def _accumulate_up():
        w1_acc_ref[...] += jnp.dot(
            k_tile_masked(lhs_ref[...], dim=1), k_tile_masked(w1_ref[...], dim=0), preferred_element_type=jnp.float32
        )
        w2_acc_ref[...] += jnp.dot(
            k_tile_masked(lhs_ref[...], dim=1), k_tile_masked(w2_ref[...], dim=0), preferred_element_type=jnp.float32
        )


    _accumulate_up()

    @pl.when(is_last_k_tile)
    def _compute_intermediate():
        hidden = jax.nn.swish(w1_acc_ref[...]) * w2_acc_ref[...]
        hidden = hidden.astype(w_down_ref.dtype)
        out_acc_ref[...] += jnp.dot(
            n_tile_masked(hidden[...], dim=1),
            n_tile_masked(w_down_ref[...], dim=0),
            preferred_element_type=jnp.float32,
        )

    @pl.when(is_last_k_tile & is_last_n_tile)
    def _store_out():
        mask = m_tile_mask()
        out_ref[...] = jnp.where(mask, out_acc_ref[...], out_ref[...]).astype(out_ref.dtype)



def ragged_dot_fused(
        lhs,
        rhs,
        group_sizes,
        tiling,
):
    """
    Args:
        lhs: jax array (m, k)
        rhs: tuple of
            w1: jax array (e, k, n)
            w2: jax array (e, k, n)
            w_down: jax array (e, n, d)
        group_sizes: jax array (e,)
        tiling: (bm, bk, bn, bd)
    """

    m, k = lhs.shape
    e, _, n = rhs[0].shape
    _, _, d = rhs[2].shape
    bm, bk, bn, bd = tiling

    assert m % bm == 0


    (group_offsets, group_ids, m_tile_ids), num_tiles = make_group_metadata(
        group_sizes=group_sizes,
        m=m,
        tm=bm,
        start_group=jnp.array([0], dtype=jnp.int32),
        num_nonzero_groups=e,
        visit_empty_groups=True
    )

    # (num_tiles, d_tiles, n_tiles, k_tiles)
    d_tiles, d_rem = _calculate_irregular_num_tiles(d, bd)
    k_tiles, k_rem = _calculate_irregular_num_tiles(k, bk)
    n_tiles, n_rem = _calculate_irregular_num_tiles(n, bn)

    grid = (d_tiles, num_tiles, n_tiles, k_tiles)

    def lhs_map(d_tile, tile_num, n_tile, k_tile, metadata):
        group_offsets, group_ids, m_tile_ids = metadata
        return m_tile_ids[tile_num], k_tile

    def up_proj_map(d_tile, tile_num, n_tile, k_tile, metadata):
        group_offsets, group_ids, m_tile_ids = metadata

        # load the correct (k, n) matrix
        return group_ids[tile_num], k_tile, n_tile

    def down_proj_map(d_tile, tile_num, n_tile, k_tile, metadata):
        group_offsets, group_ids, m_tile_ids = metadata
        return group_ids[tile_num], n_tile, d_tile

    def out_map(d_tile, tile_num, n_tile, k_tile, metadata):
        group_offsets, group_ids, m_tile_ids = metadata
        return m_tile_ids[tile_num], d_tile


    lhs_spec = pl.BlockSpec((bm, bk), lhs_map)
    w1_spec = pl.BlockSpec((None, bk, bn), up_proj_map)
    w2_spec = pl.BlockSpec((None, bk, bn), up_proj_map)
    w_down_spec = pl.BlockSpec((None, bn, bd), down_proj_map)
    outspec = pl.BlockSpec((bm, bd), out_map)

    inspecs = [
        lhs_spec,
        w1_spec,
        w2_spec,
        w_down_spec,
    ]

    scratch_shapes = [
        pltpu.VMEM((bm, bn), jnp.float32),
        pltpu.VMEM((bm, bn), jnp.float32),
        pltpu.VMEM((bm, bd), jnp.float32),
    ]

    grid_spec = pltpu.PrefetchScalarGridSpec(
        num_scalar_prefetch=1,
        grid=grid,
        in_specs=inspecs,
        out_specs=outspec,
        scratch_shapes=scratch_shapes,
    )
    compiler_params = pltpu.CompilerParams(
        dimension_semantics=("parallel", "arbitrary", "arbitrary", "arbitrary")
    )

    reduction_end_idxs = (n_tiles-1, k_tiles-1)

    out_shape = jax.ShapeDtypeStruct((m, d), lhs.dtype)

    kernel_fn = partial(
        kernel,
        k_rem=k_rem,
        n_rem=n_rem,
        bm=bm,
        reduction_end_idxs=reduction_end_idxs,
    )

    return pallas_call(
        kernel_fn,
        grid_spec=grid_spec,
        out_shape=out_shape,
        compiler_params=compiler_params,
    )(
        (group_offsets, group_ids, m_tile_ids),
        lhs,
        *rhs,
    )

### Tests

In [ ]:
def moe_ffn_reference(x, W1, W2, Wd, group_sizes):
    """Ground truth"""
    xf, W1f, W2f, Wdf = (a.astype(jnp.float32) for a in (x, W1, W2, Wd))
    outs, start = [], 0
    for g in range(len(group_sizes)):
        n = int(group_sizes[g])
        xs = xf[start:start + n]                              # [n, D]
        h  = jax.nn.swish(xs @ W1f[g]) * (xs @ W2f[g])        # [n, F]
        outs.append(h @ Wdf[g])                              # [n, D]
        start += n
    return jnp.concatenate(outs, axis=0)                      # [M, D]  (empty groups -> (0,D), fine)


def make_moe_inputs(group_sizes, d, f, dtype=jnp.float32, seed=0):
    """Normalized (1/sqrt(fan_in)) inputs"""
    G, M = len(group_sizes), int(sum(group_sizes))
    kx, k1, k2, kd = jax.random.split(jax.random.key(seed), 4)
    x  = jax.random.normal(kx, (M, d), dtype=dtype)
    W1 = jax.random.normal(k1, (G, d, f), dtype=dtype) / np.sqrt(d)
    W2 = jax.random.normal(k2, (G, d, f), dtype=dtype) / np.sqrt(d)
    Wd = jax.random.normal(kd, (G, f, d), dtype=dtype) / np.sqrt(f)
    return x, W1, W2, Wd, jnp.asarray(group_sizes, dtype=jnp.int32)


def describe_schedule(group_sizes, bn):
    gs = [int(g) for g in group_sizes]
    M  = sum(gs)
    num_tiles = (M + bn - 1) // bn
    offsets = np.concatenate([[0], np.cumsum(gs)]).astype(int)   # [G+1] row boundaries
    work = []
    for t in range(num_tiles):
        r0, r1 = t * bn, min((t + 1) * bn, M)
        for g in range(len(gs)):
            if offsets[g] < r1 and offsets[g + 1] > r0:          # group g overlaps this tile
                work.append((g, t))
    group_ids  = [g for g, _ in work]
    m_tile_ids = [t for _, t in work]
    print(f"group_sizes={gs}  bn={bn}  M={M}  num_tiles={num_tiles}  "
          f"num_active={len(work)}  straddles={len(work) - num_tiles}")
    print(f"  group_offsets = {offsets.tolist()}")
    print(f"  group_ids     = {group_ids}")
    print(f"  m_tile_ids    = {m_tile_ids}")
    for g, t in work:
        gr = t * bn + np.arange(bn)
        mask = (gr >= offsets[g]) & (gr < offsets[g + 1])
        print(f"    (g={g}, tile={t}) rows {t*bn}..{t*bn+bn-1}  mask={mask.astype(int).tolist()}")
    return offsets, group_ids, m_tile_ids, len(work)


def check_moe(x, W1, W2, Wd, group_sizes, bn, bd_in, bf, bd_out,
              dtype=jnp.float32, rtol=None, atol=None):
    tiling = (bn, bd_in, bf, bd_out)
    got = ragged_dot_fused(x, (W1, W2, Wd), group_sizes, tiling)

    expected = moe_ffn_reference(x, W1, W2, Wd, group_sizes)
    got.block_until_ready()

    if rtol is None or atol is None:
        rtol, atol = (3e-2, 3e-2) if dtype in (jnp.bfloat16, jnp.float16) else (1e-4, 1e-4)
    got32, exp32 = np.asarray(got, np.float32), np.asarray(expected, np.float32)
    np.testing.assert_allclose(got32, exp32, rtol=rtol, atol=atol)
    print(f"PASS groups={[int(g) for g in group_sizes]} d={x.shape[1]} f={W1.shape[2]} "
          f"block=(bn={bn},bd_in={bd_in},bf={bf},bd_out={bd_out}) dtype={dtype.__name__} "
          f"max_err={np.max(np.abs(got32-exp32)):.3e}")

In [ ]:
BN = 8  # token tile size for all examples

# no straddle
gsA = [16, 16]
xA, W1A, W2A, WdA, gA = make_moe_inputs(gsA, d=128, f=256)

# straddle
gsB = [12, 20]
xB, W1B, W2B, WdB, gB = make_moe_inputs(gsB, d=128, f=256)

# multi expert
gsC = [10, 6, 16]
xC, W1C, W2C, WdC, gC = make_moe_inputs(gsC, d=128, f=512)

# empty expert
gsD = [8, 0, 24]                     # M=32; expert 1 empty
xD, W1D, W2D, WdD, gD = make_moe_inputs(gsD, d=128, f=256)

for name, gs in [("A", gsA), ("B", gsB), ("C", gsC), ("D", gsD)]:
    print(f"--- Example {name} ---")
    describe_schedule(gs, BN)
    print()

# fused moe tests
check_moe(xA, W1A, W2A, WdA, gA, bn=BN, bd_in=128, bf=128, bd_out=128)
check_moe(xB, W1B, W2B, WdB, gB, bn=BN, bd_in=128, bf=128, bd_out=128)
check_moe(xC, W1C, W2C, WdC, gC, bn=BN, bd_in=128, bf=128, bd_out=128)
check_moe(xD, W1D, W2D, WdD, gD, bn=BN, bd_in=128, bf=128, bd_out=128)

--- Example A ---
group_sizes=[16, 16]  bn=8  M=32  num_tiles=4  num_active=4  straddles=0
  group_offsets = [0, 16, 32]
  group_ids     = [0, 0, 1, 1]
  m_tile_ids    = [0, 1, 2, 3]
    (g=0, tile=0) rows 0..7  mask=[1, 1, 1, 1, 1, 1, 1, 1]
    (g=0, tile=1) rows 8..15  mask=[1, 1, 1, 1, 1, 1, 1, 1]
    (g=1, tile=2) rows 16..23  mask=[1, 1, 1, 1, 1, 1, 1, 1]
    (g=1, tile=3) rows 24..31  mask=[1, 1, 1, 1, 1, 1, 1, 1]

--- Example B ---
group_sizes=[12, 20]  bn=8  M=32  num_tiles=4  num_active=5  straddles=1
  group_offsets = [0, 12, 32]
  group_ids     = [0, 0, 1, 1, 1]
  m_tile_ids    = [0, 1, 1, 2, 3]
    (g=0, tile=0) rows 0..7  mask=[1, 1, 1, 1, 1, 1, 1, 1]
    (g=0, tile=1) rows 8..15  mask=[1, 1, 1, 1, 0, 0, 0, 0]
    (g=1, tile=1) rows 8..15  mask=[0, 0, 0, 0, 1, 1, 1, 1]
    (g=1, tile=2) rows 16..23  mask=[1, 1, 1, 1, 1, 1, 1, 1]
    (g=1, tile=3) rows 24..31  mask=[1, 1, 1, 1, 1, 1, 1, 1]

--- Example C ---
group_sizes=[10, 6, 16]  bn=8  M=32  num_tiles=4  num_active=5  st

In [ ]:
# bf16
xBf, W1Bf, W2Bf, WdBf, gBf = make_moe_inputs([12, 20], d=128, f=256, dtype=jnp.bfloat16)
check_moe(xBf, W1Bf, W2Bf, WdBf, gBf, bn=BN, bd_in=128, bf=128, bd_out=128, dtype=jnp.bfloat16)

# non-divisible k
xE, W1E, W2E, WdE, gE = make_moe_inputs([12, 20], d=136, f=256)   # straddle + k remainder
check_moe(xE, W1E, W2E, WdE, gE, bn=BN, bd_in=128, bf=128, bd_out=128)

# non-divisible n
xF, W1F, W2F, WdF, gF = make_moe_inputs([10, 6, 16], d=128, f=264)  # straddle + n remainder
check_moe(xF, W1F, W2F, WdF, gF, bn=BN, bd_in=128, bf=128, bd_out=128)

# everything
xG, W1G, W2G, WdG, gG = make_moe_inputs([12, 20], d=136, f=264, dtype=jnp.bfloat16)
check_moe(xG, W1G, W2G, WdG, gG, bn=BN, bd_in=128, bf=128, bd_out=128, dtype=jnp.bfloat16)

print("All extra tests passed.")

PASS groups=[12, 20] d=128 f=256 block=(bn=8,bd_in=128,bf=128,bd_out=128) dtype=bfloat16 max_err=7.061e-03
PASS groups=[12, 20] d=136 f=256 block=(bn=8,bd_in=128,bf=128,bd_out=128) dtype=float32 max_err=0.000e+00
PASS groups=[10, 6, 16] d=128 f=264 block=(bn=8,bd_in=128,bf=128,bd_out=128) dtype=float32 max_err=0.000e+00
PASS groups=[12, 20] d=136 f=264 block=(bn=8,bd_in=128,bf=128,bd_out=128) dtype=bfloat16 max_err=7.066e-03
All extra tests passed.


# Roofline

In [ ]:
FLOPS = 1.97e14
BW = 8.2e11

def flops(b, d, f):
    return 6*b*d*f + b*f

def unfused_mem(b, d, f, e):
    inputs = 2*b*d + 6*d*f*e
    intermeds = 4*b*f + 4*b*f
    outputs = 2*b*d

    return inputs + intermeds + outputs

def fused_mem(b, d, f, e):
    inputs = 2*b*d + 6*d*f*e
    outputs = 2*b*d

    return inputs + outputs

def flops_time(b, d, f):
    return flops(b,d,f) / FLOPS

def unfused_mem_time(b, d, f,e):
    return unfused_mem(b, d, f,e) / BW

def fused_mem_time(b, d, f,e):
    return fused_mem(b,d,f,e) / BW

E = 4
B = 256
D = 128
F = 512


print(flops_time(B,D,F))

print(fused_mem_time(B,D,F,E))

print(unfused_mem_time(B,D,F,E))

print(
    unfused_mem_time(B,D,F,E)  / fused_mem_time(B,D,F,E)
)



5.11646538071066e-07
2.077970731707317e-06
3.356721951219512e-06
1.6153846153846154


# Speed Tests

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
from functools import partial

def moe_core_unfused(x, W1, W2, Wd, group_sizes):
    a = jax.lax.ragged_dot(x, W1, group_sizes)
    g = jax.lax.ragged_dot(x, W2, group_sizes)
    h = jax.nn.swish(a) * g
    return jax.lax.ragged_dot(h, Wd, group_sizes)

def moe_core_fused(x, W1, W2, Wd, group_sizes, tiling):
    return ragged_dot_fused(x, (W1, W2, Wd), group_sizes, tiling)

def make_bench_inputs(b, d, f, e, dtype=jnp.bfloat16, seed=0):
    key = jax.random.key(seed)
    kx, k1, k2, kd = jax.random.split(key, 4)
    x  = jax.random.normal(kx, (b, d), dtype=dtype)
    W1 = (jax.random.normal(k1, (e, d, f), jnp.float32) / np.sqrt(d)).astype(dtype)
    W2 = (jax.random.normal(k2, (e, d, f), jnp.float32) / np.sqrt(d)).astype(dtype)
    Wd = (jax.random.normal(kd, (e, f, d), jnp.float32) / np.sqrt(f)).astype(dtype) # nonzero
    base = b // e
    sizes = np.full(e, base, dtype=np.int64)
    sizes[: b - base * e] += 1
    return x, W1, W2, Wd, jnp.asarray(sizes, jnp.int32)

def _bench(fn, args, iters=50, warmup=10):
    for _ in range(warmup):
        fn(*args).block_until_ready()
    t0 = time.perf_counter()
    for _ in range(iters):
        out = fn(*args)
    out.block_until_ready()
    return (time.perf_counter() - t0) / iters

In [ ]:
def _predict_speedup(b, d, f, e):
    F       = flops(b, d, f) / FLOPS
    unfused = max(F, unfused_mem(b, d, f, e) / BW)
    fused   = max(F, fused_mem(b, d, f, e) / BW)
    return unfused / fused

def profile_moe(b, d, f, e, tiling, dtype=jnp.bfloat16, iters=50):
    bm = tiling[0]
    assert b % bm == 0, f"kernel needs b % bm == 0 (b={b}, bm={bm})"
    x, W1, W2, Wd, gs = make_bench_inputs(b, d, f, e, dtype)

    unfused = jax.jit(moe_core_unfused)
    fused   = jax.jit(partial(moe_core_fused, tiling=tiling))

    o_u = unfused(x, W1, W2, Wd, gs); o_u.block_until_ready()
    o_f = fused(x, W1, W2, Wd, gs);   o_f.block_until_ready()
    max_err = float(jnp.max(jnp.abs(o_u.astype(jnp.float32) - o_f.astype(jnp.float32))))

    t_u = _bench(unfused, (x, W1, W2, Wd, gs), iters)
    t_f = _bench(fused,   (x, W1, W2, Wd, gs), iters)

    Mg = b / e
    print(f"b={b:5d} (Mg={Mg:5.0f})  d={d:4d}  f={f:4d}  e={e}  tiling={tiling}")
    print(f"   unfused {t_u*1e6:8.1f} us   fused {t_f*1e6:8.1f} us   "
          f"measured {t_u/t_f:5.2f}x   predicted {_predict_speedup(b,d,f,e):5.2f}x   "
          f"max_err {max_err:.2e}")
    return t_u, t_f

In [ ]:
e = 4
b = 256
d = 256
f = 2048

# bm, bk, bn, bd
tiling = (128, 128, 128, 128)
profile_moe(b, d, f, e, tiling)

b=  256 (Mg=   64)  d= 256  f=2048  e=4  tiling=(128, 128, 128, 128)
   unfused     63.7 us   fused    136.9 us   measured  0.47x   predicted  1.33x   max_err 1.56e-02


(6.373338000230433e-05, 0.0001368720000027679)

## Sweeps to find a good setting

In [ ]:
def sweep(e=8):
    for d in [128, 256]:
        for Mg in [32, 64, 128, 256]:
            b  = Mg * e
            f  = 4 * d
            bm = min(128, Mg)
            bk = min(128, d)
            bd = min(128, d)
            bn = 128               # F
            profile_moe(b, d, f, e, (bm, bk, bn, bd))
            print()

sweep()

b=  256 (Mg=   32)  d= 128  f= 512  e=8  tiling=(32, 128, 128, 128)
   unfused     57.6 us   fused     58.6 us   measured  0.98x   predicted  1.32x   max_err 1.56e-02

b=  512 (Mg=   64)  d= 128  f= 512  e=8  tiling=(64, 128, 128, 128)
   unfused     62.1 us   fused     64.2 us   measured  0.97x   predicted  1.62x   max_err 1.56e-02

b= 1024 (Mg=  128)  d= 128  f= 512  e=8  tiling=(128, 128, 128, 128)
   unfused     60.7 us   fused     59.3 us   measured  1.02x   predicted  2.14x   max_err 1.56e-02

b= 2048 (Mg=  256)  d= 128  f= 512  e=8  tiling=(128, 128, 128, 128)
   unfused     76.8 us   fused     75.1 us   measured  1.02x   predicted  3.00x   max_err 1.56e-02

b=  256 (Mg=   32)  d= 256  f=1024  e=8  tiling=(32, 128, 128, 128)
   unfused     61.7 us   fused    116.6 us   measured  0.53x   predicted  1.16x   max_err 1.56e-02

b=  512 (Mg=   64)  d= 256  f=1024  e=8  tiling=(64, 128, 128, 128)
   unfused     88.5 us   fused    130.4 us   measured  0.68x   predicted  1.32x   max_err 

In [ ]:
# Sweep growing e
for e in [32, 64, 128, 256]:
    profile_moe(128 * e, 128, 512, e, (128, 128, 128, 128))
    print()

b= 4096 (Mg=  128)  d= 128  f= 512  e=32  tiling=(128, 128, 128, 128)
   unfused     54.8 us   fused     81.2 us   measured  0.67x   predicted  2.14x   max_err 1.56e-02

b= 8192 (Mg=  128)  d= 128  f= 512  e=64  tiling=(128, 128, 128, 128)
   unfused     75.4 us   fused    150.3 us   measured  0.50x   predicted  2.14x   max_err 1.56e-02

b=16384 (Mg=  128)  d= 128  f= 512  e=128  tiling=(128, 128, 128, 128)
   unfused    142.2 us   fused    288.0 us   measured  0.49x   predicted  2.14x   max_err 1.56e-02

b=32768 (Mg=  128)  d= 128  f= 512  e=256  tiling=(128, 128, 128, 128)
   unfused    338.5 us   fused    562.4 us   measured  0.60x   predicted  2.14x   max_err 1.56e-02



In [ ]:
# Sweep f block tile
e, Mg, d, f = 128, 128, 128, 512
b = Mg * e
for bn in [128, 256, 512]:
    profile_moe(b, d, f, e, (128, 128, bn, 128))
    print()

b=16384 (Mg=  128)  d= 128  f= 512  e=128  tiling=(128, 128, 128, 128)
   unfused    143.2 us   fused    289.8 us   measured  0.49x   predicted  2.14x   max_err 1.56e-02

b=16384 (Mg=  128)  d= 128  f= 512  e=128  tiling=(128, 128, 256, 128)
   unfused    143.2 us   fused    192.6 us   measured  0.74x   predicted  2.14x   max_err 1.56e-02

b=16384 (Mg=  128)  d= 128  f= 512  e=128  tiling=(128, 128, 512, 128)
   unfused    143.5 us   fused    137.9 us   measured  1.04x   predicted  2.14x   max_err 1.56e-02



In [ ]:
# fix 512 for f block, sweep f
d, Mg, e = 128, 128, 128
for f in [512, 1024, 2048]:
    b  = Mg * e
    bn = min(f, 512)
    profile_moe(b, d, f, e, (128, 128, bn, 128))
    print()

b=16384 (Mg=  128)  d= 128  f= 512  e=128  tiling=(128, 128, 512, 128)
   unfused    143.3 us   fused    137.9 us   measured  1.04x   predicted  2.14x   max_err 1.56e-02

b=16384 (Mg=  128)  d= 128  f=1024  e=128  tiling=(128, 128, 512, 128)
   unfused    261.0 us   fused    247.7 us   measured  1.05x   predicted  2.23x   max_err 1.56e-02

b=16384 (Mg=  128)  d= 128  f=2048  e=128  tiling=(128, 128, 512, 128)
   unfused    720.2 us   fused    450.2 us   measured  1.60x   predicted  2.28x   max_err 1.56e-02



In [ ]:
d, Mg, e = 128, 128, 16
for f in [512, 1024, 2048]:
    b  = Mg * e
    bn = min(f, 512)          # keep the f-block at 512 (efficient); more f -> more n-tiles
    profile_moe(b, d, f, e, (128, 128, bn, 128))
    print()

b= 2048 (Mg=  128)  d= 128  f= 512  e=16  tiling=(128, 128, 512, 128)
   unfused     50.8 us   fused     71.0 us   measured  0.72x   predicted  2.14x   max_err 1.56e-02

b= 2048 (Mg=  128)  d= 128  f=1024  e=16  tiling=(128, 128, 512, 128)
   unfused     68.6 us   fused     66.4 us   measured  1.03x   predicted  2.23x   max_err 1.56e-02

b= 2048 (Mg=  128)  d= 128  f=2048  e=16  tiling=(128, 128, 512, 128)
   unfused     66.4 us   fused     68.8 us   measured  0.96x   predicted  2.28x   max_err 1.56e-02



In [ ]:
# Sweep tokens per expert
b, d, f = 16384, 128, 2048
for Mg in [64, 128, 192, 256]:
    e  = b // Mg
    bm = min(128, Mg)
    profile_moe(b, d, f, e, (bm, 128, 512, 128))
    print()

b=16384 (Mg=   64)  d= 128  f=2048  e=256  tiling=(64, 128, 512, 128)
   unfused   1084.1 us   fused    790.9 us   measured  1.37x   predicted  1.65x   max_err 1.56e-02

b=16384 (Mg=  128)  d= 128  f=2048  e=128  tiling=(128, 128, 512, 128)
   unfused    719.5 us   fused    449.7 us   measured  1.60x   predicted  2.28x   max_err 1.56e-02

b=16384 (Mg=  193)  d= 128  f=2048  e=85  tiling=(128, 128, 512, 128)
   unfused    757.9 us   fused    693.7 us   measured  1.09x   predicted  2.89x   max_err 1.56e-02

b=16384 (Mg=  256)  d= 128  f=2048  e=64  tiling=(128, 128, 512, 128)
   unfused    611.6 us   fused    444.9 us   measured  1.37x   predicted  3.46x   max_err 1.56e-02



# Setting + Explanation

After some experimentation, I found a setting where we do in fact see a
measurable speedup when using my handrolled kernel, compared to simply
jitting 3 applications of `jax.ragged_dot`.

The setting is:

```
b       = 16384
d_model = 128
d_ff    = 2048
e       = 128
```

Under this setting (and completely balanced allocation) we see a **~1.60x speedup**.

Fundamentally, the handrolled kernel saves an HBM roundtrip for the
intermediate activations in the FFN block. These activations have shape
$(B, F)$. Thus we can only see a speedup in regimes where we scale either $B$ or $F$.

Furthermore, we will only see a speedup at all if we are memory bound to begin
with, rather than compute bound. If we do some roofline analysis, we
see that the intensity of this kernel is

$$I = \frac{6BF}{4B + 6Fe}$$

we can rewrite this in terms of $M_g := B / e$ (tokens per expert)

$$I = \frac{6 \cdot M_g \cdot F}{4 \cdot M_g + 6 \cdot F}$$

if we solve for $I = 240$, we get:

$$M_g = \frac{1440 \cdot F}{6 \cdot F - 960}$$

which is the compute bound / memory bound ridge (at $F = 2048$, this is equal to 260).

So we need to keep tokens / expert below 260 to stay in the memory bound regime,
which is where this kernel has a chance of beating the baseline.

If we are memory bound for our kernel, we will be memory bound for the baseline
as well (same number of FLOPs, more bytes to move around). Then the speedup will
be the ratio of bytes moved:

$$\text{speedup} = \frac{4BD + 6DFe + cBF}{4BD + 6DFe}$$

Notice that as $D$ grows large, even though this doesn't affect the intensity of
our handrolled kernel, it reduces the speedup, since the ratio of bytes moved
gets closer to 1. This is why the speedup is most apparent when $D$ is small
(128 in our setting).

You may notice there is a gap in measured vs predicted speedup. I'm guessing this
is due to several factors:

- **untuned tiling:** we sweep tiling a little bit but not a ton
- **untuned kernel:** While the kernel correctly fuses the up and down projections,
  it is not as tuned as the `ragged_dot` kernel. For example, the `ragged_dot`
  kernel skips unused experts entirely.
- **overly pessimistic on bytes moved:** It is likely that the jitted baseline
  doesn't move $8BF$ intermediate bytes. It most likely fuses some operations and
  saves some HBM roundtrips.